### Para poder usar el sistema de conversión a texto tienes que instalar algunas librerias necesarias: Ejecuta el código de la siguiente celda.

### Ejecuta cada una de las celdas de manera ordenada.

### Cuando termines de ejecutar la última celda, veras un link que te llevará a una página web. Espera 2 minutos para que el servidor responda.

In [1]:
!pip -q install streamlit edge-tts PyMuPDF python-docx pytesseract Pillow
!apt-get -qq update
!apt-get -qq install -y tesseract-ocr tesseract-ocr-spa

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package tesseract-ocr-spa.
(Reading database ... 122806 files and directories currently installed.)
Preparing to unpack .../tesseract-ocr-spa_1%3a4.1.0-2_all.deb ...
Unpacking tesseract-ocr-spa (1:4.1.0-2) ...
Setting up tesseract-ocr-spa (1:4.1.0-2) ...


In [2]:
%%writefile app.py
import streamlit as st
import edge_tts
import asyncio
import fitz
import uuid
import re
import io
import pytesseract

from PIL import Image
from pathlib import Path
from docx import Document

# =============================================
# CONFIGURACIÓN
# =============================================

st.set_page_config(
    page_title="Lector de Documentos",
    page_icon="🔊",
    layout="centered"
)

CARPETA_AUDIOS = Path("audios")
CARPETA_AUDIOS.mkdir(exist_ok=True)

VOCES = {
    "Mujer - Español Latino": "es-MX-DaliaNeural",
    "Hombre - Español Latino": "es-MX-JorgeNeural",
    "Mujer - Español España": "es-ES-ElviraNeural",
    "Hombre - Español España": "es-ES-AlvaroNeural",
    "Mujer - Inglés": "en-US-JennyNeural",
    "Hombre - Inglés": "en-US-GuyNeural",
    "Mujer - Italiano": "it-IT-ElsaNeural",
    "Hombre - Italiano": "it-IT-DiegoNeural"
}

VELOCIDADES = {
    "0.75x - Lenta": "-25%",
    "1.00x - Normal": "+0%",
    "1.25x - Rápida": "+25%",
    "1.50x - Muy rápida": "+50%"
}

# =============================================
# LECTURA DE ARCHIVOS
# =============================================

def leer_txt(archivo):
    contenido = archivo.getvalue()

    for codificacion in ["utf-8", "cp1252", "latin-1"]:
        try:
            return contenido.decode(codificacion)
        except UnicodeDecodeError:
            continue

    return contenido.decode("utf-8", errors="replace")


def leer_pdf(archivo):
    contenido = []

    with fitz.open(
        stream=archivo.getvalue(),
        filetype="pdf"
    ) as documento:

        for numero, pagina in enumerate(documento, start=1):
            texto = pagina.get_text("text").strip()

            if texto:
                contenido.append(
                    f"--- Página {numero} ---\n{texto}"
                )

    return "\n\n".join(contenido)


def leer_docx(archivo):
    documento = Document(io.BytesIO(archivo.getvalue()))
    contenido = []

    for parrafo in documento.paragraphs:
        if parrafo.text.strip():
            contenido.append(parrafo.text.strip())

    for tabla in documento.tables:
        for fila in tabla.rows:
            celdas = [
                celda.text.strip()
                for celda in fila.cells
                if celda.text.strip()
            ]

            if celdas:
                contenido.append(". ".join(celdas))

    return "\n\n".join(contenido)


def leer_imagen(archivo):
    imagen = Image.open(io.BytesIO(archivo.getvalue()))

    if imagen.mode not in ("RGB", "L"):
        imagen = imagen.convert("RGB")

    texto = pytesseract.image_to_string(
        imagen,
        lang="spa",
        config="--oem 3 --psm 6"
    )

    return texto.strip()


def extraer_texto(archivo):
    extension = Path(archivo.name).suffix.lower()

    if extension == ".txt":
        return leer_txt(archivo)

    if extension == ".pdf":
        return leer_pdf(archivo)

    if extension == ".docx":
        return leer_docx(archivo)

    if extension in [".jpg", ".jpeg", ".png"]:
        return leer_imagen(archivo)

    raise ValueError("Formato no compatible.")


# =============================================
# PREPARAR TEXTO
# =============================================

def limpiar_texto(texto):
    texto = re.sub(r"--- Página \d+ ---", " ", texto)
    texto = re.sub(r"\n{3,}", "\n\n", texto)
    texto = re.sub(r"[ \t]+", " ", texto)

    return texto.strip()


def dividir_texto(texto, limite=2500):
    texto = limpiar_texto(texto)
    oraciones = re.split(r"(?<=[.!?])\s+", texto)

    bloques = []
    actual = ""

    for oracion in oraciones:
        if len(oracion) > limite:
            if actual.strip():
                bloques.append(actual.strip())
                actual = ""

            for i in range(0, len(oracion), limite):
                bloques.append(oracion[i:i + limite])

            continue

        if len(actual) + len(oracion) + 1 <= limite:
            actual += " " + oracion
        else:
            if actual.strip():
                bloques.append(actual.strip())

            actual = oracion

    if actual.strip():
        bloques.append(actual.strip())

    return bloques


# =============================================
# GENERACIÓN DE AUDIO
# =============================================

async def generar_audio_edge(texto, voz, velocidad):
    bloques = dividir_texto(texto)

    if not bloques:
        raise ValueError("No existe texto para generar audio.")

    ruta_final = CARPETA_AUDIOS / f"{uuid.uuid4()}.mp3"
    temporales = []

    try:
        for numero, bloque in enumerate(bloques):
            temporal = (
                CARPETA_AUDIOS /
                f"temp_{uuid.uuid4()}_{numero}.mp3"
            )

            temporales.append(temporal)

            comunicacion = edge_tts.Communicate(
                bloque,
                voz,
                rate=velocidad
            )

            await comunicacion.save(str(temporal))

        with open(ruta_final, "wb") as salida:
            for temporal in temporales:
                with open(temporal, "rb") as entrada:
                    salida.write(entrada.read())

        return ruta_final, len(bloques)

    finally:
        for temporal in temporales:
            temporal.unlink(missing_ok=True)


def generar_audio(texto, voz, velocidad):
    return asyncio.run(
        generar_audio_edge(texto, voz, velocidad)
    )


# =============================================
# INTERFAZ STREAMLIT
# =============================================

st.title("🔊 Lector de Documentos")

st.write(
    "Convierte documentos e imágenes en audio MP3 "
    "utilizando voces naturales."
)

st.divider()

st.subheader("1. Seleccionar documento")

archivo = st.file_uploader(
    "Selecciona un documento o imagen",
    type=["txt", "pdf", "docx", "jpg", "jpeg", "png"]
)

if archivo is not None:

    identificador = (
        archivo.name,
        archivo.size,
        archivo.file_id
    )

    if st.session_state.get("archivo_actual") != identificador:
        st.session_state["archivo_actual"] = identificador
        st.session_state.pop("texto_editable", None)
        st.session_state.pop("audio_generado", None)

    try:
        with st.spinner("Extrayendo texto..."):
            texto_original = extraer_texto(archivo)

        st.success("Documento procesado correctamente.")

        col1, col2 = st.columns(2)

        with col1:
            st.write(f"**Archivo:** {archivo.name}")

        with col2:
            st.write(
                f"**Formato:** {Path(archivo.name).suffix.upper()}"
            )

        st.divider()
        st.subheader("2. Texto extraído")

        if texto_original.strip():

            if "texto_editable" not in st.session_state:
                st.session_state["texto_editable"] = texto_original

            texto = st.text_area(
                "Puedes corregir el texto antes de generar el audio:",
                key="texto_editable",
                height=400
            )

            col1, col2, col3 = st.columns(3)

            with col1:
                st.metric("Palabras", len(texto.split()))

            with col2:
                st.metric("Caracteres", len(texto))

            with col3:
                st.metric("Fragmentos", len(dividir_texto(texto)))

            st.divider()

            st.subheader("3. Configuración de voz")

            nombre_voz = st.selectbox(
                "Voz",
                list(VOCES.keys())
            )

            velocidad_nombre = st.selectbox(
                "Velocidad de lectura",
                list(VELOCIDADES.keys()),
                index=1
            )

            st.divider()
            st.subheader("4. Generar audio")

            if st.button(
                "🔊 Generar MP3",
                type="primary",
                use_container_width=True
            ):

                if not texto.strip():
                    st.warning("No existe texto para convertir.")

                else:
                    try:
                        with st.spinner("Generando audio..."):
                            ruta, cantidad = generar_audio(
                                texto,
                                VOCES[nombre_voz],
                                VELOCIDADES[velocidad_nombre]
                            )

                        datos_audio = ruta.read_bytes()
                        ruta.unlink(missing_ok=True)

                        st.session_state["audio_generado"] = datos_audio
                        st.session_state["cantidad_bloques"] = cantidad

                    except Exception as error:
                        st.error(
                            f"Error al generar audio: {error}"
                        )

            if "audio_generado" in st.session_state:
                st.success("Audio generado correctamente.")

                st.caption(
                    f"Se procesaron "
                    f"{st.session_state['cantidad_bloques']} fragmentos."
                )

                st.audio(
                    st.session_state["audio_generado"],
                    format="audio/mp3"
                )

                st.download_button(
                    "💾 Guardar MP3",
                    data=st.session_state["audio_generado"],
                    file_name=f"{Path(archivo.name).stem}.mp3",
                    mime="audio/mpeg",
                    use_container_width=True
                )

        else:
            st.warning(
                "No se encontró texto dentro del documento."
            )

            if Path(archivo.name).suffix.lower() == ".pdf":
                st.info(
                    "Este PDF podría estar escaneado. "
                    "El OCR para PDF se incorporará posteriormente."
                )

    except Exception as error:
        st.error(f"Error al procesar el documento: {error}")

else:
    st.info(
        "Selecciona un archivo TXT, PDF, DOCX, JPG, JPEG o PNG."
    )

Writing app.py


### Una vez que ejecutes esta celda, veras un link en la parte inferior. Ingresa al link y espera como 2 minutos, es el tiempo que el servidor tada en responde.

In [3]:
import subprocess
import time
import re
import os

# Descargar cloudflared si todavía no existe
if not os.path.exists("/usr/local/bin/cloudflared"):
    subprocess.run(
        [
            "wget", "-q",
            "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
            "-O", "/usr/local/bin/cloudflared"
        ],
        check=True
    )

    subprocess.run(
        ["chmod", "+x", "/usr/local/bin/cloudflared"],
        check=True
    )

# Iniciar Streamlit
streamlit_log = open("/content/streamlit.log", "w")

streamlit_proceso = subprocess.Popen(
    [
        "streamlit", "run", "app.py",
        "--server.port", "8501",
        "--server.address", "127.0.0.1",
        "--server.headless", "true"
    ],
    stdout=streamlit_log,
    stderr=subprocess.STDOUT
)

time.sleep(5)

# Crear túnel temporal
cloudflared_proceso = subprocess.Popen(
    [
        "/usr/local/bin/cloudflared",
        "tunnel",
        "--url", "http://127.0.0.1:8501",
        "--no-autoupdate"
    ],
    stdout=open("/content/cloudflared.log", "w"),
    stderr=subprocess.STDOUT
)

url = None

for intento in range(30):
    time.sleep(2)

    with open("/content/cloudflared.log", "r") as archivo_log:
        contenido = archivo_log.read()

    coincidencia = re.search(
        r"https://[a-zA-Z0-9-]+\.trycloudflare\.com",
        contenido
    )

    if coincidencia:
        url = coincidencia.group(0)
        break

if url:
    print("APLICACIÓN DISPONIBLE:")
    print(url)
else:
    print("No se pudo obtener el enlace público.")
    print("Revisa los registros de Cloudflare y Streamlit.")

APLICACIÓN DISPONIBLE:
https://thumbnail-diamond-decided-perfume.trycloudflare.com
